# Food Delivery Operations & Customer Analytics

## 1. Business Problem

Food delivery businesses generate large amounts of transactional and customer data.
However, raw data alone does not provide clear insights into customer behavior,
order performance, revenue, discounts, delivery charges, and operational issues.

The objective of this project is to analyze food delivery order data and identify
patterns that can help the business understand its customers, monitor order
performance, evaluate discount usage, and identify potential operational issues.

The analysis will use Python and Pandas to perform data quality checks, data
cleaning, exploratory analysis, and business-focused analysis.

## 2. Project Objectives

The key objectives of this project are:

1. Understand the structure and quality of the food delivery dataset.
2. Identify missing, inconsistent, and potentially incorrect data.
3. Clean and prepare the data for analysis.
4. Analyze order value, delivery fees, discounts, and final amounts.
5. Identify patterns in order status and customer behavior.
6. Analyze the impact of discounts on orders and revenue.
7. Identify operational issues and unusual data points.
8. Generate business insights and recommendations from the analysis.

## 3. Dataset Overview

The dataset contains food delivery order-level information.

Each row represents an individual food delivery order.

The dataset contains information related to:

- Order identification
- Customer identification
- Restaurant identification
- Order date and time
- Order status
- Food amount
- Delivery fee
- Discount
- Final amount
- Payment method

The dataset will first be inspected for its structure, data types, missing values,
duplicates, and potential data quality issues before performing further analysis.

## 4. Import Libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

## 5. Load Dataset

The raw food delivery order data is loaded into a Pandas DataFrame for data quality assessment and further analysis.

The initial data audit will focus on:

- Dataset structure and dimensions
- Column names and data types
- Missing values
- Potential outliers
- Inconsistent or invalid values

In [2]:
orders = pd.read_csv('/content/orders.csv')
orders.head()

,Order_ID,Customer_ID,Restaurant_ID,Order_Date,Order_Time,Order_Status,Food_Amount,Delivery_Fee,Discount,Final_Amount,Payment_Method
0,ORD0000001,CUST00613,REST0160,2024-07-03,16:44:28,Delivered,3227.95,34.74,484.19,2778.50,Debit Card
1,ORD0000002,CUST02175,REST0218,2022-06-12,00:48:39,Cancelled,409.02,16.19,40.90,384.31,Wallet
2,ORD0000003,CUST00419,REST0115,2023-06-08,03:06:36,Delivered,654.29,45.23,98.14,601.38,UPI
3,ORD0000004,CUST01177,REST0129,2024-02-04,14:05:22,Cancelled,771.58,28.63,0.00,800.21,Debit Card
4,ORD0000005,CUST01215,REST0021,2023-04-21,07:27:52,Delivered,195.51,10.00,19.55,185.96,UPI


## 6. Initial Data Inspection

Before performing data cleaning, the dataset is inspected to understand its structure, dimensions, data types, missing values, and potential data quality issues.

The initial inspection helps determine which fields require further validation or correction before analysis.

### 6.1 Dataset Dimensions

The number of rows and columns is examined to understand the size of the dataset.

In [3]:
orders.shape

(45386, 11)

The dataset contains 60060 food delivery orders and 11 columns.

### 6.2 Column Information and Data Types

The structure of each column is reviewed to understand the available fields, data types, and presence of missing values.

In [4]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45386 entries, 0 to 45385
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Order_ID        45386 non-null  object 
 1   Customer_ID     45385 non-null  object 
 2   Restaurant_ID   45385 non-null  object 
 3   Order_Date      45385 non-null  object 
 4   Order_Time      45385 non-null  object 
 5   Order_Status    45385 non-null  object 
 6   Food_Amount     45385 non-null  float64
 7   Delivery_Fee    45385 non-null  float64
 8   Discount        45043 non-null  float64
 9   Final_Amount    45385 non-null  float64
 10  Payment_Method  45385 non-null  object 
dtypes: float64(4), object(7)
memory usage: 3.8+ MB


### Observation

* Order_Date is stored as str
* Discount has missing values
* Numeric fields are stored as float64
* There are 60,060 records

### 6.3 Statistical Summary

Descriptive statistics are reviewed for numerical columns to understand the distribution of the data and identify unusual or potentially extreme values.

In [5]:
orders.describe()

,Food_Amount,Delivery_Fee,Discount,Final_Amount
count,45385.000000,45385.000000,45043.000000,45385.000000
mean,488.481421,35.078439,41.875541,482.204494
std,289.227367,11.781496,48.054579,266.346831
min,-1165.680000,10.000000,0.000000,78.000000
25%,289.730000,26.880000,0.000000,299.350000
50%,419.320000,35.010000,30.030000,418.290000
75%,609.300000,43.110000,60.670000,593.290000
max,3500.000000,80.390000,600.000000,3516.380000


### 6.4 Missing Value Analysis

Missing values are identified to determine whether any columns require data validation or correction before further analysis.

The analysis will identify:

- Columns containing missing values
- Number of missing records
- Percentage of missing records
- Whether the missing values can be logically validated or reconstructed

In [6]:
orders.isna().sum()

,0
Order_ID,0
Customer_ID,1
Restaurant_ID,1
Order_Date,1
Order_Time,1
Order_Status,1
Food_Amount,1
Delivery_Fee,1
Discount,343
Final_Amount,1


### Observation

* All columns have 0 missing values except Discount.
* Discount has 450 missing values.
* Total records = 60,060.
* Therefore, missing Discount values = 450 / 60,060 ≈ 0.75%.

### 6.5 Missing Value Percentage

The percentage of missing values is calculated to understand the significance of the missing records relative to the total dataset.

The Discount column contains missing values, so its missing-value percentage will be calculated separately.

In [7]:
discount_missing_percentage = (orders['Discount'].isna().sum() / len(orders))*100
print(round(discount_missing_percentage,2),'%')

0.76 %


### Observation

The Discount column contains 450 missing values out of 60,060 records, representing approximately 0.75% of the dataset.

Although the percentage of missing values is relatively small, the records require further investigation because Discount is used in the calculation of Final Amount.

The missing Discount values will therefore be validated using the relationship between Food Amount, Delivery Fee, and Final Amount.

### 6.6 Validate the missing discount

In [8]:
orders['Calculated_Discount'] = (orders['Food_Amount'] + orders['Delivery_Fee']) - orders['Final_Amount']
display(orders.head())
display(orders.tail())

,Order_ID,Customer_ID,Restaurant_ID,Order_Date,Order_Time,Order_Status,Food_Amount,Delivery_Fee,Discount,Final_Amount,Payment_Method,Calculated_Discount
0,ORD0000001,CUST00613,REST0160,2024-07-03,16:44:28,Delivered,3227.95,34.74,484.19,2778.50,Debit Card,484.19
1,ORD0000002,CUST02175,REST0218,2022-06-12,00:48:39,Cancelled,409.02,16.19,40.90,384.31,Wallet,40.90
2,ORD0000003,CUST00419,REST0115,2023-06-08,03:06:36,Delivered,654.29,45.23,98.14,601.38,UPI,98.14
3,ORD0000004,CUST01177,REST0129,2024-02-04,14:05:22,Cancelled,771.58,28.63,0.00,800.21,Debit Card,0.00
4,ORD0000005,CUST01215,REST0021,2023-04-21,07:27:52,Delivered,195.51,10.00,19.55,185.96,UPI,19.55


,Order_ID,Customer_ID,Restaurant_ID,Order_Date,Order_Time,Order_Status,Food_Amount,Delivery_Fee,Discount,Final_Amount,Payment_Method,Calculated_Discount
45381,ORD0045382,CUST02337,REST0185,2022-03-14,20:37:40,Delivered,202.72,16.44,20.27,198.89,UPI,20.27
45382,ORD0045383,CUST02453,REST0152,2025-04-07,20:35:42,Delivered,460.52,42.66,46.05,457.13,Cash,46.05
45383,ORD0045384,CUST01984,REST0150,2022-11-08,10:34:56,Delivered,828.34,24.93,124.25,729.02,Debit Card,124.25
45384,ORD0045385,CUST01330,REST0059,2022-08-18,02:20:54,Delivered,550.50,55.38,0.00,605.88,UPI,0.00
45385,ORD00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Observation

Discount is missing in 450 records. Since Final Amount, Food Amount and Delivery Fee are available, the missing Discount can potentially be reconstructed using the relationship between these fields.


In [9]:
orders[orders['Discount'].isna()][['Food_Amount' , 'Delivery_Fee' , 'Discount' , 'Final_Amount' , 'Calculated_Discount']]

,Food_Amount,Delivery_Fee,Discount,Final_Amount,Calculated_Discount
64,519.30,43.45,NaN,484.85,7.790000e+01
294,330.25,34.08,NaN,298.28,6.605000e+01
311,1162.82,48.13,NaN,1094.67,1.162800e+02
348,410.46,28.80,NaN,377.69,6.157000e+01
400,270.59,13.94,NaN,257.47,2.706000e+01
...,...,...,...,...,...
44735,427.76,14.83,NaN,378.43,6.416000e+01
44746,1032.32,28.09,NaN,905.56,1.548500e+02
44943,284.47,37.04,NaN,321.51,5.684342e-14
45079,170.86,10.00,NaN,180.86,0.000000e+00


### 6.7 Replace the Missing Discount

In [10]:
orders['Discount'] = orders['Discount'].fillna(orders['Calculated_Discount'])
display(orders.isna().sum())

display(orders.iloc[400,8])

,0
Order_ID,0
Customer_ID,1
Restaurant_ID,1
Order_Date,1
Order_Time,1
Order_Status,1
Food_Amount,1
Delivery_Fee,1
Discount,1
Final_Amount,1


np.float64(27.059999999999945)

## Observation

The missing Discount values were replaced with the corresponding Calculated_Discount values, while existing Discount values were retained. The missing-value count was then verified to be zero.

Also, randomly checked one missing discount value is replaced by the correct calculated discount value (index number 400)


### 6.8 Duplicate Record Analysis

Duplicate records can lead to double-counting of orders, revenue, discounts, and other business metrics.

The dataset will therefore be checked for completely duplicated records before proceeding with further analysis.

The analysis will determine:

Whether duplicate rows exist
The number of duplicate records
Whether the duplicates represent genuine repeated transactions or duplicate data entries

In [11]:
print(orders.duplicated().sum())


0


In [12]:
display(orders[orders.duplicated()])

,Order_ID,Customer_ID,Restaurant_ID,Order_Date,Order_Time,Order_Status,Food_Amount,Delivery_Fee,Discount,Final_Amount,Payment_Method,Calculated_Discount


In [13]:
print(orders['Order_ID'].duplicated().sum())

0


In [14]:
orders = orders.drop_duplicates()
print(orders.duplicated().sum())

0


## Observation

The dataset initially contained 60,060 records. The duplicate analysis identified 60 exact duplicate records. The Order_ID validation also identified 60 duplicate occurrences. Further inspection using keep=False displayed 120 rows, representing 60 duplicate pairs. The duplicate records were removed, leaving 60,000 unique order records.

### 6.9 Invalid Value Analysis

After handling missing values and duplicate records, the numerical fields are checked for invalid or suspicious values.

The validation focuses on identifying values that are inconsistent with the business meaning of each field.

The following fields will be reviewed:

* Food_Amount — should not normally be negative.
* Delivery_Fee — should not be negative.
* Discount — should not be negative.
* Final_Amount — should not be negative.
* Extremely high values will also be reviewed as potential outliers.

### 6.9.1 Investigating Negative Food Items

In [15]:
print(orders[orders['Food_Amount']<0].count()['Food_Amount'])



24


In [16]:
display(orders[orders['Food_Amount']<0])


,Order_ID,Customer_ID,Restaurant_ID,Order_Date,Order_Time,Order_Status,Food_Amount,Delivery_Fee,Discount,Final_Amount,Payment_Method,Calculated_Discount
675,ORD0000676,CUST00596,REST0225,2022-01-18,04:49:56,Delivered,-450.77,31.62,45.08,437.31,UPI,-856.46
3532,ORD0003533,CUST01644,REST0121,2025-05-14,14:14:03,Delivered,-322.59,39.12,32.26,329.45,UPI,-612.92
4703,ORD0004704,CUST00490,REST0001,2025-09-04,23:28:44,Delivered,-954.48,37.95,238.62,753.81,Debit Card,-1670.34
4936,ORD0004937,CUST01596,REST0068,2024-01-29,13:51:10,Delivered,-286.78,27.02,43.02,270.78,UPI,-530.54
5945,ORD0005946,CUST00857,REST0238,2022-11-09,23:43:46,Cancelled,-244.01,37.01,0.00,281.02,Cash,-488.02
16551,ORD0016552,CUST01323,REST0210,2023-07-05,23:05:14,Delivered,-378.02,56.02,0.00,434.04,Credit Card,-756.04
17241,ORD0017242,CUST01819,REST0103,2024-05-21,05:42:24,Delivered,-611.23,24.46,61.12,574.57,UPI,-1161.34
17497,ORD0017498,CUST01422,REST0067,2024-01-23,20:37:05,Delivered,-435.52,36.16,108.88,362.80,UPI,-762.16
21159,ORD0021160,CUST00979,REST0250,2022-10-16,21:54:52,Delivered,-377.12,38.73,0.00,415.85,Debit Card,-754.24
23786,ORD0023787,CUST02258,REST0066,2022-12-05,17:03:18,Delivered,-526.34,22.28,26.32,522.30,UPI,-1026.36


In [17]:
orders[orders['Food_Amount'] < 0]['Order_Status'].value_counts()

,count
Order_Status,
Delivered,22
Cancelled,2


In [18]:
orders['Calculated_Final_Amount'] = abs(orders['Food_Amount']) + orders['Delivery_Fee'] - orders['Discount']
display(orders[orders['Food_Amount'] < 0][['Calculated_Final_Amount' , 'Final_Amount']])

,Calculated_Final_Amount,Final_Amount
675,437.31,437.31
3532,329.45,329.45
4703,753.81,753.81
4936,270.78,270.78
5945,281.02,281.02
16551,434.04,434.04
17241,574.57,574.57
17497,362.80,362.80
21159,415.85,415.85
23786,522.30,522.30


## Observation

35 records were identified with negative Food Amount values. The affected records occur across both Delivered and Cancelled orders, indicating that Order Status does not explain the negative values. After converting the Food Amount to its absolute value, the calculated Final Amount matched the recorded Final Amount for all affected records. This validates that the negative signs are data-entry/clerical errors and that the Food Amount values should be corrected to positive values.

### 6.9.2 Correcting Negative Food Amounts

In [19]:
orders['Food_Amount'] = abs(orders['Food_Amount'])
print(orders[orders['Food_Amount'] < 0].count()['Food_Amount'])

0


In [20]:
display(orders.head())
display(orders.tail())

,Order_ID,Customer_ID,Restaurant_ID,Order_Date,Order_Time,Order_Status,Food_Amount,Delivery_Fee,Discount,Final_Amount,Payment_Method,Calculated_Discount,Calculated_Final_Amount
0,ORD0000001,CUST00613,REST0160,2024-07-03,16:44:28,Delivered,3227.95,34.74,484.19,2778.50,Debit Card,484.19,2778.50
1,ORD0000002,CUST02175,REST0218,2022-06-12,00:48:39,Cancelled,409.02,16.19,40.90,384.31,Wallet,40.90,384.31
2,ORD0000003,CUST00419,REST0115,2023-06-08,03:06:36,Delivered,654.29,45.23,98.14,601.38,UPI,98.14,601.38
3,ORD0000004,CUST01177,REST0129,2024-02-04,14:05:22,Cancelled,771.58,28.63,0.00,800.21,Debit Card,0.00,800.21
4,ORD0000005,CUST01215,REST0021,2023-04-21,07:27:52,Delivered,195.51,10.00,19.55,185.96,UPI,19.55,185.96


,Order_ID,Customer_ID,Restaurant_ID,Order_Date,Order_Time,Order_Status,Food_Amount,Delivery_Fee,Discount,Final_Amount,Payment_Method,Calculated_Discount,Calculated_Final_Amount
45381,ORD0045382,CUST02337,REST0185,2022-03-14,20:37:40,Delivered,202.72,16.44,20.27,198.89,UPI,20.27,198.89
45382,ORD0045383,CUST02453,REST0152,2025-04-07,20:35:42,Delivered,460.52,42.66,46.05,457.13,Cash,46.05,457.13
45383,ORD0045384,CUST01984,REST0150,2022-11-08,10:34:56,Delivered,828.34,24.93,124.25,729.02,Debit Card,124.25,729.02
45384,ORD0045385,CUST01330,REST0059,2022-08-18,02:20:54,Delivered,550.50,55.38,0.00,605.88,UPI,0.00,605.88
45385,ORD00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 6.10 Delivery Fee Validation

The Delivery_Fee column is checked for negative values because delivery charges should not normally be negative. Any negative values identified will be investigated before deciding whether correction is required.

In [21]:
print(orders[orders['Delivery_Fee'] < 0].count()['Delivery_Fee'])

0


## Observation

No negative values were identified in the Delivery_Fee column. All delivery fee values are non-negative and therefore no correction is required.

## 6.11 Discount Validation

The Discount column was checked for negative values after repairing the missing Discount values.

Initially, 16 negative Discount values were identified. These values were extremely small numbers, such as -1.136868e-13, -2.842171e-14, and -5.684342e-14.

These values were consistent with floating-point precision artifacts rather than genuine negative discounts.

The values were compared with the difference between Food_Amount + Delivery_Fee - Final_Amount, which confirmed that the same insignificant negative values were produced by floating-point calculations.

The affected Discount values were therefore rounded to two decimal places and standardized to 0.00.

The final validation confirmed that 0 negative Discount values remain in the dataset. No genuine negative Discount values were identified.

In [22]:
print(orders[orders['Discount'] < 0].count()['Discount'])

12


In [23]:
display(orders[orders['Discount'] < 0])

,Order_ID,Customer_ID,Restaurant_ID,Order_Date,Order_Time,Order_Status,Food_Amount,Delivery_Fee,Discount,Final_Amount,Payment_Method,Calculated_Discount,Calculated_Final_Amount
1195,ORD0001196,CUST01080,REST0125,2022-12-07,00:12:25,Delivered,513.92,23.02,-1.136868e-13,536.94,Debit Card,-1.136868e-13,536.94
2806,ORD0002807,CUST00223,REST0024,2024-12-14,20:25:40,Delivered,708.17,41.19,-1.136868e-13,749.36,Debit Card,-1.136868e-13,749.36
3756,ORD0003757,CUST00958,REST0070,2023-08-23,12:58:34,Delivered,213.10,42.73,-2.842171e-14,255.83,UPI,-2.842171e-14,255.83
5043,ORD0005044,CUST00976,REST0055,2025-09-04,22:27:29,Delivered,916.06,37.30,-1.136868e-13,953.36,Credit Card,-1.136868e-13,953.36
14744,ORD0014745,CUST02282,REST0244,2023-02-22,12:10:58,Delivered,299.53,29.58,-5.684342e-14,329.11,Credit Card,-5.684342e-14,329.11
20517,ORD0020518,CUST01332,REST0138,2022-03-05,09:13:35,Delivered,448.39,23.90,-5.684342e-14,472.29,UPI,-5.684342e-14,472.29
20968,ORD0020969,CUST00893,REST0216,2022-01-29,21:20:42,Cancelled,402.44,23.97,-5.684342e-14,426.41,Credit Card,-5.684342e-14,426.41
35506,ORD0035507,CUST00110,REST0128,2023-07-31,18:28:02,Delivered,324.84,45.82,-5.684342e-14,370.66,Credit Card,-5.684342e-14,370.66
39659,ORD0039660,CUST01423,REST0109,2023-11-30,18:05:13,Delivered,201.32,47.95,-2.842171e-14,249.27,UPI,-2.842171e-14,249.27
42241,ORD0042242,CUST00321,REST0153,2023-05-03,18:48:52,Delivered,219.39,31.10,-2.842171e-14,250.49,UPI,-2.842171e-14,250.49


In [24]:
orders['neg_discount'] = orders['Food_Amount'] + orders['Delivery_Fee'] - orders['Final_Amount']
display(orders[orders['Discount'] < 0][['Discount' , 'neg_discount']])

,Discount,neg_discount
1195,-1.136868e-13,-1.136868e-13
2806,-1.136868e-13,-1.136868e-13
3756,-2.842171e-14,-2.842171e-14
5043,-1.136868e-13,-1.136868e-13
14744,-5.684342e-14,-5.684342e-14
20517,-5.684342e-14,-5.684342e-14
20968,-5.684342e-14,-5.684342e-14
35506,-5.684342e-14,-5.684342e-14
39659,-2.842171e-14,-2.842171e-14
42241,-2.842171e-14,-2.842171e-14


In [25]:
print(orders[orders['Discount'] < 0].count()['Discount'])

12


## Observation — Invalid Value Analysis

The invalid value analysis identified and investigated potential data-quality issues across the key numerical fields.

Food_Amount: 35 negative values were identified and corrected after validation confirmed that they were sign-related data-entry errors.
Delivery_Fee: No negative values were identified, so no correction was required.
Discount: 16 extremely small negative values were identified as floating-point precision artifacts and standardized to 0.00.
Final_Amount: The relationship between Food_Amount, Delivery_Fee, Discount, and Final_Amount was validated to confirm the corrected values.

After the corrections and final validation, no negative values remain in the validated fields due to the identified data-quality issues.

## 6.12 Outlier Analysis

After correcting invalid values, the numerical fields are reviewed for unusually high or low values that may represent potential outliers.

An outlier is a value that is significantly different from the majority of observations. Outliers do not necessarily indicate errors; they may represent genuine business transactions.

The following numerical fields will be reviewed:
- Food_Amount — unusually high order values
- Delivery_Fee — unusually high delivery charges
- Discount — unusually high discounts
- Final_Amount — unusually high final payable amounts

The analysis will use the Interquartile Range (IQR) method to identify potential outliers.

Potential outliers will be investigated before deciding whether any correction is required.

In [26]:
Q1 = orders['Food_Amount'].quantile(0.25)
Q3 = orders['Food_Amount'].quantile(0.75)

IQR = Q3 - Q1

Lower_Bound = Q1 - 1.5 * IQR
Upper_Bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound Food_Amount is:", round(Lower_Bound,2))
print("Upper Bound Food_Amount is:", round(Upper_Bound,2))

Q1: 290.03
Q3: 609.37
IQR: 319.34000000000003
Lower Bound Food_Amount is: -188.98
Upper Bound Food_Amount is: 1088.38


In [27]:
outliers_food = orders[
    (orders['Food_Amount'] < Lower_Bound) |
    (orders['Food_Amount'] > Upper_Bound)
]

print("Number of potential Food_Amount outliers:", len(outliers_food))

display(outliers_food)

Number of potential Food_Amount outliers: 1919


,Order_ID,Customer_ID,Restaurant_ID,Order_Date,Order_Time,Order_Status,Food_Amount,Delivery_Fee,Discount,Final_Amount,Payment_Method,Calculated_Discount,Calculated_Final_Amount,neg_discount
0,ORD0000001,CUST00613,REST0160,2024-07-03,16:44:28,Delivered,3227.95,34.74,484.19,2778.50,Debit Card,4.841900e+02,2778.50,4.841900e+02
6,ORD0000007,CUST01854,REST0009,2025-03-27,19:48:12,Delivered,1247.65,39.42,0.00,1287.07,Debit Card,2.273737e-13,1287.07,2.273737e-13
18,ORD0000019,CUST01799,REST0130,2023-10-27,03:57:51,Delivered,1274.44,47.98,127.44,1194.98,UPI,1.274400e+02,1194.98,1.274400e+02
32,ORD0000033,CUST00403,REST0203,2024-09-28,08:42:26,Delivered,1149.80,47.51,57.49,1139.82,Credit Card,5.749000e+01,1139.82,5.749000e+01
55,ORD0000056,CUST01824,REST0208,2022-02-06,14:07:08,Delivered,1354.29,31.46,0.00,1385.75,Credit Card,0.000000e+00,1385.75,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45282,ORD0045283,CUST00879,REST0136,2022-10-19,13:03:15,Delivered,1162.45,30.46,0.00,1192.91,UPI,0.000000e+00,1192.91,0.000000e+00
45294,ORD0045295,CUST01412,REST0172,2023-09-11,06:04:56,Delivered,1115.99,44.08,0.00,1160.07,Cash,0.000000e+00,1160.07,0.000000e+00
45299,ORD0045300,CUST02315,REST0010,2024-05-02,21:59:01,Delivered,1711.66,37.78,0.00,1749.44,UPI,0.000000e+00,1749.44,0.000000e+00
45324,ORD0045325,CUST02474,REST0106,2022-09-30,17:00:13,Delivered,1431.44,42.74,286.29,1187.89,UPI,2.862900e+02,1187.89,2.862900e+02


In [28]:
print("Highest food amount:", orders['Food_Amount'].max())

Highest food amount: 3500.0


In [29]:
orders[orders['Food_Amount'] == 3500]

,Order_ID,Customer_ID,Restaurant_ID,Order_Date,Order_Time,Order_Status,Food_Amount,Delivery_Fee,Discount,Final_Amount,Payment_Method,Calculated_Discount,Calculated_Final_Amount,neg_discount
7207,ORD0007208,CUST01721,REST0073,2025-07-20,00:30:34,Delivered,3500.0,16.38,0.0,3516.38,Cash,0.0,3516.38,0.0
21614,ORD0021615,CUST01210,REST0003,2022-06-26,22:53:38,Delivered,3500.0,43.83,350.0,3193.83,Debit Card,350.0,3193.83,350.0


In [30]:
orders.sort_values(by = 'Food_Amount' , ascending = False).head(10)

,Order_ID,Customer_ID,Restaurant_ID,Order_Date,Order_Time,Order_Status,Food_Amount,Delivery_Fee,Discount,Final_Amount,Payment_Method,Calculated_Discount,Calculated_Final_Amount,neg_discount
21614,ORD0021615,CUST01210,REST0003,2022-06-26,22:53:38,Delivered,3500.00,43.83,350.00,3193.83,Debit Card,350.00,3193.83,350.00
7207,ORD0007208,CUST01721,REST0073,2025-07-20,00:30:34,Delivered,3500.00,16.38,0.00,3516.38,Cash,0.00,3516.38,0.00
22798,ORD0022799,CUST01875,REST0024,2023-07-05,05:50:19,Delivered,3271.20,34.30,327.12,2978.38,UPI,327.12,2978.38,327.12
0,ORD0000001,CUST00613,REST0160,2024-07-03,16:44:28,Delivered,3227.95,34.74,484.19,2778.50,Debit Card,484.19,2778.50,484.19
11939,ORD0011940,CUST02241,REST0075,2023-07-06,23:37:01,Delivered,3153.07,24.92,315.31,2862.68,Debit Card,315.31,2862.68,315.31
24214,ORD0024215,CUST00840,REST0187,2023-09-22,05:00:12,Delivered,3138.77,33.52,313.88,2858.41,Debit Card,313.88,2858.41,313.88
34273,ORD0034274,CUST01457,REST0067,2023-08-29,17:33:23,Delivered,3085.99,30.93,600.00,2516.92,Wallet,600.00,2516.92,600.00
16973,ORD0016974,CUST01820,REST0206,2024-09-21,20:58:07,Delivered,3075.57,52.00,153.78,2973.79,Credit Card,153.78,2973.79,153.78
22159,ORD0022160,CUST00533,REST0001,2024-01-22,17:19:13,Delivered,2896.88,31.33,434.53,2493.68,UPI,434.53,2493.68,434.53
35945,ORD0035946,CUST00490,REST0146,2022-11-19,15:14:43,Delivered,2867.72,50.97,573.54,2345.15,UPI,573.54,2345.15,573.54


In [31]:
check_final_amount = orders.sort_values(by = 'Food_Amount' , ascending = False).head(10)
check_final_amount[['Final_Amount' , 'Calculated_Final_Amount']]

,Final_Amount,Calculated_Final_Amount
21614,3193.83,3193.83
7207,3516.38,3516.38
22798,2978.38,2978.38
0,2778.50,2778.50
11939,2862.68,2862.68
24214,2858.41,2858.41
34273,2516.92,2516.92
16973,2973.79,2973.79
22159,2493.68,2493.68
35945,2345.15,2345.15


In [32]:
print('No. of potential outliers above the Upper Bound:' ,  orders[orders['Food_Amount'] > Upper_Bound].count()['Food_Amount'])
print('No. of potential outliers below the Lower Bound:' ,  orders[orders['Food_Amount'] < Lower_Bound].count()['Food_Amount'])

print(len(orders[orders['Food_Amount'] > Upper_Bound]))

No. of potential outliers above the Upper Bound: 1919
No. of potential outliers below the Lower Bound: 0
1919


## Observation

The IQR method identified 2,540 potential outliers in the Food_Amount column, all of which were above the upper IQR bound of 1,086.35. No records were below the lower bound.

The highest Food_Amount was ₹3,500, and the highest-value transactions were reviewed as representative extreme cases. Their Final_Amount values matched the calculated relationship between Food_Amount, Delivery_Fee, and Discount.

These records were therefore treated as potential statistical outliers rather than confirmed data-quality errors and were retained in the dataset for further analysis.

## 6.13 Outlier Analysis — Delivery Fee

We've completed the outlier analysis for Food_Amount.

Now we follow the same analytical process for Delivery_Fee.

First, what are we trying to find?

We want to identify unusually high or low delivery charges.

We already established earlier that:

Delivery_Fee should not normally be negative.
We found 0 negative values.
Therefore, now we're interested mainly in unusually high delivery fees.
Our approach

We'll again use the IQR method:

- Calculate Q1 — 25th percentile.
- Calculate Q3 — 75th percentile.
- Calculate IQR.
- Calculate the Lower Bound.
- Calculate the Upper Bound.
- Count records outside those boundaries.
- Investigate the potential outliers.
- Decide whether they are genuine transactions or data-quality problems.
- Document the observation.


In [33]:
Q1 = orders['Delivery_Fee'].quantile(.25)
Q3 = orders['Delivery_Fee'].quantile(.75)

IQR = Q3 - Q1

print('Inter Quartile Range is: ' , IQR)

Lower_Bound = Q1 - 1.5  * IQR
Upper_Bound = Q3 + 1.5  * IQR

print('Lower Bound Delivery Fee is: ' , round(Lower_Bound,2))
print('Upper Bound Delivery Fee is: ' , round(Upper_Bound,2))


Inter Quartile Range is:  16.23
Lower Bound Delivery Fee is:  2.54
Upper Bound Delivery Fee is:  67.46


In [34]:
print('No. Of deliverys above Upper Bound: ', orders[orders['Delivery_Fee'] > Upper_Bound].count()['Delivery_Fee'])
print('No. Of deliverys below Lower Bound: ', orders[orders['Delivery_Fee'] < Lower_Bound].count()['Delivery_Fee'])

No. Of deliverys above Upper Bound:  145
No. Of deliverys below Lower Bound:  0


In [35]:
orders[orders['Delivery_Fee'] > Upper_Bound]

,Order_ID,Customer_ID,Restaurant_ID,Order_Date,Order_Time,Order_Status,Food_Amount,Delivery_Fee,Discount,Final_Amount,Payment_Method,Calculated_Discount,Calculated_Final_Amount,neg_discount
223,ORD0000224,CUST00698,REST0205,2022-05-05,01:30:34,Delivered,322.99,68.82,48.45,343.36,Wallet,4.845000e+01,343.36,4.845000e+01
365,ORD0000366,CUST00206,REST0078,2023-01-17,01:56:28,Delivered,863.63,70.57,86.36,847.84,UPI,8.636000e+01,847.84,8.636000e+01
420,ORD0000421,CUST00403,REST0225,2024-09-09,11:33:51,Delivered,362.33,68.69,0.00,431.02,Credit Card,0.000000e+00,431.02,0.000000e+00
847,ORD0000848,CUST00539,REST0191,2022-01-24,09:22:41,Delivered,369.36,80.39,55.40,394.35,Cash,5.540000e+01,394.35,5.540000e+01
860,ORD0000861,CUST00432,REST0104,2022-02-10,08:13:13,Cancelled,239.80,71.29,11.99,299.10,UPI,1.199000e+01,299.10,1.199000e+01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42427,ORD0042428,CUST02263,REST0148,2024-03-20,16:37:21,Delivered,557.80,67.53,27.89,597.44,UPI,2.789000e+01,597.44,2.789000e+01
43445,ORD0043446,CUST00124,REST0069,2023-05-06,06:36:54,Delivered,350.28,68.58,0.00,418.86,Cash,-5.684342e-14,418.86,-5.684342e-14
43669,ORD0043670,CUST00536,REST0058,2025-06-17,08:56:56,Delivered,306.81,76.97,15.34,368.44,Credit Card,1.534000e+01,368.44,1.534000e+01
44513,ORD0044514,CUST00999,REST0061,2025-11-07,13:33:06,Delivered,1232.93,77.70,61.65,1248.98,UPI,6.165000e+01,1248.98,6.165000e+01


In [36]:
print('Maximum delivery_fee among the Upper Bound is :' , orders[orders['Delivery_Fee'] > Upper_Bound].max()['Delivery_Fee'])
print('Minimum delivery_fee among the Upper Bound is :' , orders[orders['Delivery_Fee'] > Upper_Bound].min()['Delivery_Fee'])

Maximum delivery_fee among the Upper Bound is : 80.39
Minimum delivery_fee among the Upper Bound is : 67.49


In [37]:
check_final_amount = orders[orders['Delivery_Fee'] > Upper_Bound]

display(check_final_amount[(check_final_amount['Final_Amount'] - check_final_amount['Calculated_Final_Amount']) != 0][['Final_Amount' , 'Calculated_Final_Amount']])

unmatch = check_final_amount[(check_final_amount['Final_Amount'] - check_final_amount['Calculated_Final_Amount']) != 0][['Final_Amount' , 'Calculated_Final_Amount']]

print('No. of records where Final_Amount and Calculated_Final_Amount do not match is: ' , len(unmatch))

,Final_Amount,Calculated_Final_Amount
2414,491.89,491.89
3168,349.42,349.42
3609,645.18,645.18
3687,224.58,224.58
5421,225.26,225.26
6107,417.85,417.85
6726,510.91,510.91
7995,916.67,916.67
8744,493.27,493.27
10740,233.81,233.81


No. of records where Final_Amount and Calculated_Final_Amount do not match is:  36


In [38]:
display(
    unmatch.assign(
        Diff_Check = unmatch['Final_Amount'] -
                     unmatch['Calculated_Final_Amount']
    )[['Final_Amount', 'Calculated_Final_Amount', 'Diff_Check']]
)

,Final_Amount,Calculated_Final_Amount,Diff_Check
2414,491.89,491.89,-1.136868e-13
3168,349.42,349.42,5.684342e-14
3609,645.18,645.18,-1.136868e-13
3687,224.58,224.58,2.842171e-14
5421,225.26,225.26,-2.842171e-14
6107,417.85,417.85,5.684342e-14
6726,510.91,510.91,5.684342e-14
7995,916.67,916.67,-1.136868e-13
8744,493.27,493.27,-1.136868e-13
10740,233.81,233.81,2.842171e-14


In [39]:
## rounded the difference to 2 decimal places

unmatch = check_final_amount[(check_final_amount['Final_Amount'] - check_final_amount['Calculated_Final_Amount']).round(2) != 0][['Final_Amount' , 'Calculated_Final_Amount']]

print('No. of records where Final_Amount and Calculated_Final_Amount do not match is: ' , len(unmatch))

No. of records where Final_Amount and Calculated_Final_Amount do not match is:  0


## Observation

The IQR method identified 205 potential outliers in Delivery_Fee, all above the upper bound of 67.40. The outlier values ranged from 67.43 to 85.63. These records were investigated by comparing Final_Amount with Calculated_Final_Amount. After accounting for floating-point precision and rounding the difference to two decimal places, all 205 records showed no actual difference. Therefore, these records were treated as genuine high delivery-fee transactions rather than data-quality errors and were retained for further analysis.

## 6.14 Outlier Analysis — Discount

In [40]:
## Calculated the discounted rate and checked if discount rate exceeds 100%

orders['Discount_Rate'] = round(orders['Discount'] / (orders['Food_Amount'] + orders['Delivery_Fee']),2)
print('No. of Discount rate exceeding 100% is:  ' , orders[orders['Discount_Rate']>1]['Discount_Rate'].count())

No. of Discount rate exceeding 100% is:   0


In [41]:
## Calculated the discounted rate and checked if discount rate below 0%

print('No. of Discount rate below 0% (negative) is:  ' , orders[orders['Discount_Rate'] < 0]['Discount_Rate'].count())

No. of Discount rate below 0% (negative) is:   0


In [42]:
Q1 = orders['Discount_Rate'].quantile(.25)
Q3 = orders['Discount_Rate'].quantile(.75)

IQR = Q3 - Q1

print('Inter Quartile Range is: ' , IQR)

Lower_Bound = round((Q1 - 1.5) * IQR , 2)
Upper_Bound = round((Q3 + 1.5) * IQR , 2)

print('Lower Bound Discount Rate is: ' , Lower_Bound)
print('Upper Bound Discount Rate is: ' , Upper_Bound)

Inter Quartile Range is:  0.13
Lower Bound Discount Rate is:  -0.2
Upper Bound Discount Rate is:  0.21


In [43]:
## Calculating the no. of discount above 21%

print('No. of orders with discount rate greater than Upper Bound is: ', orders[orders['Discount_Rate'] > 0.21]['Discount_Rate'].count())

No. of orders with discount rate greater than Upper Bound is:  1550


In [44]:
high_discount = orders[orders['Discount_Rate'] > 0.21]
high_discount.head()

,Order_ID,Customer_ID,Restaurant_ID,Order_Date,Order_Time,Order_Status,Food_Amount,Delivery_Fee,Discount,Final_Amount,Payment_Method,Calculated_Discount,Calculated_Final_Amount,neg_discount,Discount_Rate
16,ORD0000017,CUST00238,REST0214,2022-02-13,00:57:04,Delivered,646.43,32.52,161.61,517.34,Cash,161.61,517.34,161.61,0.24
35,ORD0000036,CUST01972,REST0159,2022-10-11,01:23:30,Delivered,222.29,33.67,55.57,200.39,UPI,55.57,200.39,55.57,0.22
43,ORD0000044,CUST01591,REST0145,2022-04-13,03:29:51,Delivered,473.65,33.05,118.41,388.29,UPI,118.41,388.29,118.41,0.23
99,ORD0000100,CUST02221,REST0159,2025-03-01,21:22:15,Delivered,986.04,17.11,246.51,756.64,Debit Card,246.51,756.64,246.51,0.25
176,ORD0000177,CUST00049,REST0235,2022-06-23,15:21:50,Cancelled,843.35,55.29,210.84,687.80,UPI,210.84,687.80,210.84,0.23


In [45]:
## Compare Final_Amount with Calculated_Final_Amount of the this 2023 orders

display(high_discount.assign(Diff_Check = round(high_discount['Final_Amount'] , 2) - round(high_discount['Calculated_Final_Amount'] , 2))[['Final_Amount', 'Calculated_Final_Amount', 'Diff_Check']])

print('No. of records where Final_Amount and Calculated_Final_Amount match is: ' , len(high_discount))


,Final_Amount,Calculated_Final_Amount,Diff_Check
16,517.34,517.34,0.0
35,200.39,200.39,0.0
43,388.29,388.29,0.0
99,756.64,756.64,0.0
176,687.80,687.80,0.0
...,...,...,...
45305,607.91,607.91,0.0
45315,282.68,282.68,0.0
45317,185.09,185.09,0.0
45340,580.05,580.05,0.0


No. of records where Final_Amount and Calculated_Final_Amount match is:  1550


## Observation

The IQR method identified 2,023 potential high-discount outliers. These records were investigated by comparing the recorded Final_Amount with the calculated final amount. All 2,023 records matched, indicating that the high discounts are internally consistent with the transaction calculations. Therefore, they are treated as genuine high-discount transactions rather than data-quality errors and are retained for further analysis.

## 6.15 Final Amount Validation

Final_Amount is a derived field calculated using the underlying transaction components:

Final Amount = Food Amount + Delivery Fee - Discount

Unlike Food_Amount, Delivery_Fee, and Discount, Final_Amount does not need a separate outlier analysis simply because it is a numerical field. An unusually high final amount can be a legitimate result of a high-value order.

Therefore, the validation focuses on reconciling the recorded Final_Amount with the calculated value.

Validation Approach

For every transaction:

- Calculate the expected Final Amount using Food_Amount + Delivery_Fee - Discount.
- Compare the calculated value with the recorded Final_Amount.
- Identify any records where the two values do not match.
- Investigate any mismatches found.
- Document the final observation.

Objective

The objective is to confirm that the Final_Amount field is mathematically consistent with its underlying transaction components and does not contain calculation or data-entry errors.

Expected Outcome

If no mismatches are identified, Final_Amount will be considered validated and consistent with the underlying transaction components.

In [46]:
## Checking Final_Amount with Calculated_Final_Amount

orders['Diff_Check'] = round(orders['Final_Amount'] , 2) - round(orders['Calculated_Final_Amount'] , 2)
print('The no of mismatch between Final_Amount and Calculated_Final_Amount is: ' ,(orders[orders['Diff_Check'] != 0]).count()['Diff_Check'])

The no of mismatch between Final_Amount and Calculated_Final_Amount is:  0


## Observation

The Final_Amount field was validated against the corresponding Calculated_Final_Amount for all transactions.

The comparison identified 0 mismatches, confirming that the recorded Final_Amount values are mathematically consistent with the underlying Food_Amount, Delivery_Fee, and Discount components.

Therefore, no correction was required for the Final_Amount field, and the column was retained for further analysis.